In [ ]:
import import_ipynb
from Modelo import VotoInvalidoException, PistaInvalidaException, ConfiguracionPartida, Partida, Ganador
from Servicios import ValidadorPalabraProhibida, ValidadorLongitud

class ControladorVotacion:
    def __init__(self, vista, partida):
        self._vista = vista
        self._partida = partida

    def ejecutar(self):
        activos = self._partida.jugadores_activos
        pistas = self._partida.ronda_actual.pistas
        for votante in activos:

            candidatos = [j for j in activos if j is not votante]
            nombres = [j.nombre for j in candidatos]
            self._vista.limpiar_pantalla()
            self._vista.esperar_enter(f"Pasa el computador a {votante.nombre}. Enter para votar...")
            self._vista.mostrar_pistas(pistas)   # <- las pistas se ven mientras se vota
            while True:
                posicion = self._vista.elegir_opcion(f"{votante.nombre}, ¿quién crees que es el impostor?", nombres)
                try:
                    self._partida.votar(votante, candidatos[posicion])
                    break
                except VotoInvalidoException as error:
                    self._vista.mostrar_error(str(error))
  
        self._vista.limpiar_pantalla()
        acusado = self._partida.cerrar_votacion()
        if acusado is None:
            self._vista.mostrar_mensaje("Hubo empate: nadie fue eliminado.")
        else:
            self._vista.mostrar_mensaje(f"{acusado.nombre} fue el más votado y queda eliminado.")
        return acusado


class ControladorRonda:
    def __init__(self, vista, partida, controlador_votacion, configuracion):
        self._vista = vista
        self._partida = partida
        self._controlador_votacion = controlador_votacion
        self._configuracion = configuracion

    def ejecutar(self):
        ronda = self._partida.ronda_actual
        self._vista.mostrar_mensaje(f"\n     RONDA {ronda.numero} ")

        for jugador in self._partida.jugadores_activos:
            if ronda.pistas:
                self._vista.mostrar_pistas(ronda.pistas)
            while True:
                texto = self._vista.pedir_texto(f"{jugador.nombre}, escribe tu pista:")
                try:
                    self._partida.dar_pista(jugador, texto)
                    break
                except PistaInvalidaException as error:
                    self._vista.mostrar_error(str(error))
        self._partida.avanzar_fase()
        self._vista.mostrar_pistas(ronda.pistas)
        self._vista.mostrar_mensaje("\nDebatan quién creen que es el impostor.")
        self._vista.esperar_debate(self._configuracion.tiempo_debate)

        self._partida.avanzar_fase()
        eliminado = self._controlador_votacion.ejecutar()

        self._partida.avanzar_fase()
        return eliminado


class ControladorPartida:
    
    RONDAS_DEFECTO = 3
    DEBATE_DEFECTO = 30
    ULTIMA_OPORTUNIDAD_DEFECTO = True
    LONGITUD_PISTA_DEFECTO = 20

    def __init__(self, vista, repositorio, asignador, contador_votos):
        self._vista = vista
        self._repositorio = repositorio
        self._asignador = asignador
        self._contador_votos = contador_votos

    def _registrar_jugadores(self):
        opciones = [str(n) for n in range(3, 11)]
        cantidad = self._vista.elegir_opcion("¿Cuántos jugadores van a jugar?", opciones) + 3
        jugadores = []
        usados = set()
        for i in range(1, cantidad + 1):
            while True:
                nombre = self._vista.pedir_texto(f"Nombre del jugador {i}:")
                if not nombre:
                    self._vista.mostrar_error("El nombre no puede estar vacío.")
                elif nombre.lower() in usados:
                    self._vista.mostrar_error("Ese nombre ya está en uso.")
                else:
                    break
            usados.add(nombre.lower())
            jugadores.append(Jugador(i, nombre))
        return jugadores

    def _pedir_configuracion(self, categoria):
        personalizar = self._vista.pedir_si_no(
            "¿Quieres personalizar la configuración? (Por defecto: 3 rondas, 30 segundos de debate, última oportunidad activada, longitud máxima de pista 20)")
        if not personalizar:
            return ConfiguracionPartida(self.RONDAS_DEFECTO, self.DEBATE_DEFECTO, categoria,
                                        self.ULTIMA_OPORTUNIDAD_DEFECTO, self.LONGITUD_PISTA_DEFECTO)
        while True:
            rondas = self._vista.pedir_entero("Número de rondas (1-10):", 1, 10)
            tiempo = self._vista.pedir_entero("Tiempo de debate en segundos (6-600):", 6, 600)
            ultima = self._vista.pedir_si_no("¿El impostor tendrá una última oportunidad?")
            longitud = self._vista.pedir_entero("Longitud máxima de una pista (5-100):", 5, 100)
            configuracion = ConfiguracionPartida(rondas, tiempo, categoria, ultima, longitud)
            try:
                configuracion.validar_configuracion()
                return configuracion
            except ValueError as error:
                self._vista.mostrar_error(str(error))

    def _revelar_roles(self, partida, categoria):
        for jugador in partida.jugadores:
            self._vista.limpiar_pantalla()
            self._vista.esperar_enter(f"Pasa el dispositivo a {jugador.nombre}. Enter para ver tu rol...")
            self._vista.mostrar_rol_privado(jugador, categoria)
            self._vista.esperar_enter("Enter para ocultarlo y pasar al siguiente...")
            self._vista.limpiar_pantalla()

    def jugar(self):
        self._vista.mostrar_mensaje("EL IMPOSTOR")
        jugadores = self._registrar_jugadores()

        categorias = self._repositorio.listar_categorias()
        posicion = self._vista.elegir_opcion("Elige la categoría:", categorias)
        categoria = categorias[posicion]

        configuracion = self._pedir_configuracion(categoria)
        palabra = self._repositorio.obtener_palabra_aleatoria(categoria)
        validadores = [ValidadorPalabraProhibida(palabra),
                       ValidadorLongitud(configuracion.longitud_maxima_pista)]
        partida = Partida(jugadores, palabra, configuracion,
                          self._asignador, self._contador_votos, validadores)
        partida.iniciar()
        self._revelar_roles(partida, categoria)

        controlador_votacion = ControladorVotacion(self._vista, partida)
        controlador_ronda = ControladorRonda(self._vista, partida, controlador_votacion, configuracion)
        while True:
            controlador_ronda.ejecutar()
            resultado = partida.determinar_ganador()
            if (resultado is not None and resultado.ganador == Ganador.CIVILES
                    and configuracion.ultima_oportunidad):
                intento = self._vista.pedir_texto(
                    f"{resultado.impostor.nombre}, te descubrieron. Última oportunidad, intenta adivinar la palabra:")
                resultado = partida.determinar_ganador(intento)
            if resultado is not None:
                break
            partida.nueva_ronda()
        self._vista.mostrar_resultado(resultado)
        return resultado